**FER+ model comparison.** Run the four scripts first (mlp.py, cnn.py, qwen_zeroshot.py, qwen_finetune.py), or run_all.py which runs them all. Each one saves a JSON file in results/, and this notebook turns those files into the tables and graphs for the slides.

In [ ]:
import glob
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import common

ORDER = ["mlp", "cnn_no_miopen", "cnn", "qwen_zeroshot", "qwen_finetune"]
COLORS = dict(zip(ORDER, ["#1f77b4", "#ffbb78", "#ff7f0e", "#2ca02c", "#9467bd"]))
SETS = ["train", "validation", "test", "internet"]

results = [json.load(open(path)) for path in glob.glob("results/*.json")]
results = [r for r in results if all(name in r for name in SETS)]
results.sort(key=lambda r: ORDER.index(r["model"]) if r["model"] in ORDER else len(ORDER))
names = [r["model"] for r in results]
colors = [COLORS.get(name, "gray") for name in names]
print(names)

**Four sets of pictures.** Every model is scored separately on four sets:\n\n- train: a fixed sample of the training pictures, which the trained models learned from\n- validation: the pictures used to choose the best epoch\n- test: pictures that played no part in training\n- internet: the eight extra pictures from the internet, far too few to measure a model\n\nA comparison is only fair when every model was scored on the same number of pictures in a set.

In [ ]:
for name in SETS:
    counts = {r["model"]: r[name]["images"] for r in results}
    note = "" if len(set(counts.values())) == 1 else "   <-- not the same for every model"
    print(f"{name:10s} {counts}{note}")

In [ ]:
rows = []
for r in results:
    row = {"model": r["model"]}
    for name in SETS:
        row[f"{name}_accuracy"] = r[name]["accuracy"]
    for name in SETS[:3]:
        row[f"{name}_macro_f1"] = r[name]["macro_f1"]
    for column in ["train_time_s", "inference_ms_per_image", "params", "size_mb", "peak_gpu_mb", "device"]:
        row[column] = r[column]
    rows.append(row)
table = pd.DataFrame(rows).set_index("model")
table.to_csv("results/summary.csv")
table

In [ ]:
accuracy = pd.DataFrame({r["model"]: {name: r[name]["accuracy"] for name in SETS} for r in results}, index=SETS)
macro_f1 = pd.DataFrame({r["model"]: {name: r[name]["macro_f1"] for name in SETS[:3]} for r in results}, index=SETS[:3])
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [4, 3]})
for ax, scores, title in [(axes[0], accuracy, "Accuracy on each set of pictures"), (axes[1], macro_f1, "Macro F1 on each set of pictures")]:
    scores.plot.bar(ax=ax, color=colors, ylim=(0, 1.1), rot=0, width=0.85, title=title)
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.2f", fontsize=7)
plt.tight_layout()
plt.savefig("results/quality.png", dpi=150)

In [ ]:
def bar_chart(ax, column, title, log=False):
    values = table[column]
    bars = ax.bar(values.index, values, color=colors)
    ax.bar_label(bars, labels=[f"{v:.3f}" if v < 10 else f"{v:,.0f}" for v in values])
    ax.set_title(title)
    if log:
        ax.set_yscale("log")


fig, axes = plt.subplots(2, 2, figsize=(10, 8))
bar_chart(axes[0, 0], "train_time_s", "Training time (seconds)")
bar_chart(axes[0, 1], "inference_ms_per_image", "Inference time (ms per image, log scale)", log=True)
bar_chart(axes[1, 0], "params", "Parameters (log scale)", log=True)
bar_chart(axes[1, 1], "peak_gpu_mb", "Peak GPU memory (MB)")
plt.tight_layout()
plt.savefig("results/cost.png", dpi=150)

In [ ]:
per_class = pd.DataFrame({r["model"]: r["test"]["per_class_f1"] for r in results})
ax = per_class.plot.bar(figsize=(10, 4), color=colors, ylim=(0, 1), rot=0)
ax.set_title("F1 per emotion on the test set")
plt.tight_layout()
plt.savefig("results/per_class_f1.png", dpi=150)

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 5), squeeze=False)
for ax, r in zip(axes[0], results):
    matrix = np.array(r["test"]["confusion_matrix"])
    ax.imshow(matrix, cmap="Blues")
    ax.set_title(f"{r['model']} on the test set")
    ax.set_xticks(range(len(common.EMOTIONS)), common.EMOTIONS, rotation=45)
    ax.set_yticks(range(len(common.EMOTIONS)), common.EMOTIONS)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    for i in range(len(common.EMOTIONS)):
        for j in range(len(common.EMOTIONS)):
            text_color = "white" if matrix[i, j] > matrix.max() / 2 else "black"
            ax.text(j, i, matrix[i, j], ha="center", va="center", fontsize=7, color=text_color)
plt.tight_layout()
plt.savefig("results/confusion_matrices.png", dpi=150)

**The eight internet pictures.** A small demo, not a score: one picture per emotion, shown the way the models saw it (grayscale, 48 x 48 pixels).

In [ ]:
internet_images, _ = common.load_internet_images()
fig, axes = plt.subplots(1, len(common.EMOTIONS), figsize=(18, 2.6 + 0.3 * len(results)))
for label, ax in enumerate(axes):
    ax.imshow(internet_images[label], cmap="gray", vmin=0, vmax=255)
    ax.set_title(common.EMOTIONS[label])
    ax.axis("off")
    for row, r in enumerate(results):
        answer = r["internet"]["predictions"][label]
        guess = common.EMOTIONS[answer] if answer >= 0 else "unclear"
        mark, color = ("\u2713", "#1a7f37") if answer == label else ("\u2717", "#b3261e")
        ax.text(0.5, -0.08 - 0.14 * row, f"{mark} {r['model']}: {guess}", transform=ax.transAxes, ha="center", va="top", fontsize=8, color=color)
plt.savefig("results/internet_images.png", dpi=150, bbox_inches="tight")